<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-02-23

| Package | Version |
|---------|---------|
| **nnsight** | **0.6.0a1** |
| Python | 3.12.3 |
| torch | 2.10.0+cu128 |
| transformers | 5.2.0 |

</details>


# Logit Lens

## Introduction

🔍 Logit Lens is a powerful tool that grants us a simplified (yet insightful) understanding of the inner workings of transformer models.

We can estimate the model's guess for the output after each computational step by applying a softmax function to each layer's output. Unlike traditional approaches focusing on *how* beliefs are updated within a step, with Logit Lens we gain a glimpse into *what* output the model is predicting at each processing step.

📗 Read more about Logit Lens from nostalgebraist’s blog post on LessWrong, [here](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)

💻 You can find a Colab version of our tutorial [here](https://colab.research.google.com/github/ndif-team/nnsight-website/blob/docs/source/notebooks/tutorials/probing/logit_lens.ipynb), or nostalgebraist’s original code [here](https://colab.research.google.com/drive/1-nOE-Qyia3ElM17qrdoHAtGmLCPUZijg?usp=sharing)

🖼️ Try out Logit Lens interface at [workbench.ndif.us](workbench.ndif.us)

## Setup

If using Colab, install NNsight:
```
!pip install -U nnsight
```

In [1]:
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    !pip install -U nnsight git+https://github.com/AdamBelfki3/nnsightful.git

Import libraries and load GPT-2 model.

In [2]:
# Import libraries
from IPython.display import clear_output
from nnsight import TransformersModel
import torch
from IPython.display import clear_output

clear_output()

In [3]:
# Load gpt2
model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

## GPT-2 Model Architecture

Let's take a look at GPT-2's architecture. GPT-2 has 12 layers, accessed as `model.transformer.h`.

In [4]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
  (generator): Generator(
    (streamer): Streamer()
  )
)


## Apply Logit Lens



To apply logit lens, we collect activations at each layer's output, apply layer normalization (`model.transformer.ln_f`), and then process through the model's head (`model.lm_head`) to get the logits. Next, we apply the softmax to the logits to obtain output token probabilities.

By observing different layers' output token probabilities, logit lens provides insights into the model's confidence throughout processing steps.

In [5]:
prompt= "The Eiffel Tower is in the city of"
layers = model.transformer.h
probs_layers = []

with model.trace() as tracer:
    with tracer.invoke(prompt) as invoker:
      # store input tokens
      input_tokens = model.inputs.save()
      for layer_idx, layer in enumerate(layers):
          # Process layer output through the model's head and layer normalization
          layer_output = model.lm_head(model.transformer.ln_f(layer.output))

          # Apply softmax to obtain probabilities and save the result
          probs = torch.nn.functional.softmax(layer_output, dim=-1).save()
          probs_layers.append(probs)

probs = torch.cat([probs for probs in probs_layers])

# Find the maximum probability and corresponding tokens for each position
max_probs, tokens = probs.max(dim=-1)

# Decode token IDs to words for each layer
words = [[model.tokenizer.decode(t.cpu()).encode("unicode_escape").decode() for t in layer_tokens]
    for layer_tokens in tokens]

# Access the 'input_ids' attribute of the invoker object to get the input words
input_words = [model.tokenizer.decode(t) for t in input_tokens[1]['input_ids'][0]]

Next, we'll visualize the prediction of the GPT-2 model while processing the string *`'The Eiffel Tower is in the city of'`* and we’ll explore the interpretations of each layer within the GPT2Block, gaining insights into what each layer believes could be the next word for every input word.

## Interactive Logit Lens

Now that you've built the logit lens by hand, we'll reach for two tools from the NDIF ecosystem for how to use it in practice. 
1. `nnterp`'s `StandardizedTransformer` wraps the nnsight `TransformersModel` and enforces a uniform naming convention across model families, so the same code runs on any model unchanged. 
2. `nnsightful` is an interpretability toolkit that packages logit lens into a single call and ships an interactive visualization you can embed in a notebook or webpage. `logit_lens(model, prompt, ...)` returns a `LogitLensData` object, and `.display()` renders the data into an interactivative visualization. —see the [nnsightful repo](https://github.com/AdamBelfki3/nnsightful.git)

In [6]:
from nnterp import StandardizedTransformer
from nnsightful import logit_lens

model = StandardizedTransformer("openai-community/gpt2")

logit_lens(model, "The Eiffel Tower is located in the city of").display(full_height=True)

The horizontal axis indexes the layers, zero-indexed from 0 to 11. The vertical axis indexes the tokens of the prompt. The top guess for each token, according to the model’s activations at a given layer, is printed in each cell. The colors show the probability associated with the top guess.